# 16. Temporal Forecasting and Early-Warning Model Development

This is a **separate forecasting experiment**, not a relabeling of the event severity classifier.
It estimates whether a county records at least one political violence event with 3+ fatalities
in the seven calendar days starting at each Monday origin. All 47 counties appear every week,
including weeks with no recorded events. Only complete target windows are retained.

Historical export coverage ends **7 September 2025**. This experiment cannot issue current alerts.
ACLED `timestamp` is not treated as a historical first-seen timestamp. The default seven-day
reporting delay is an assumption; it does not reconstruct what was known in real time or
eliminate revision bias. Repeat with 0 and 14 days as sensitivity analyses. Zero event counts
mean no recorded incident, not verified absence of violence.

Features use dates strictly before origin minus the delay, with full 7-, 30-, and 90-calendar-day
windows. No same-day or future event characteristics enter predictors. Historical high-event
counts and the target include Battles, Violence against civilians, and Explosions/Remote violence
records whose disorder type contains Political violence. Context counts exclude Strategic developments.
Repeated actor rows count once per event ID when forecasting fields agree; conflicting IDs are rejected.
Election, neighbour, and external indicators await independently documented inputs.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

FORECAST_ROOT = Path.cwd()
sys.path.insert(0, str(FORECAST_ROOT / "src"))
from severity_service.forecasting import county_weeks, run_research

FORECAST_COVERAGE_END = "2025-09-07"  # Last day of this export; never substitute today's date.
REPORTING_DELAY_DAYS = 7
forecast_events = pd.read_csv(FORECAST_ROOT / "ACLED_Kenya_1997-2025.csv", low_memory=False)
forecast_panel = county_weeks(forecast_events, coverage_end=FORECAST_COVERAGE_END,
                             reporting_delay_days=REPORTING_DELAY_DAYS)
display(forecast_panel.head())
display(forecast_panel.groupby(forecast_panel.origin.dt.year).target.agg(["size", "mean"]))
assert forecast_panel.groupby("origin").county.nunique().eq(47).all()
assert (forecast_panel.label_end_exclusive <= pd.Timestamp(FORECAST_COVERAGE_END) + pd.Timedelta(days=1)).all()


## 16.1 Chronological selection, calibration, and untouched test

Compare a prevalence baseline, Logistic Regression, Random Forest, and XGBoost.
Select the model by mean rolling-origin average precision over 2019, 2020, and 2021,
each trained only on earlier fully available label windows. Fit the selected model before
2022, calibrate on 2022, choose a positive-class F1 threshold on 2023, and evaluate once
on 2024 onwards. At each boundary purge seven-day labels plus the assumed reporting delay.
All counties share Monday origins and forecast windows do not overlap.

Report precision, recall, macro F1, average precision (PR-AUC summary), ROC-AUC, Brier score,
false positive rate, false discovery rate, reliability bins, and county/year breakdowns.
The threshold is a research choice, not an operational cost-based alert policy.
Calibration may improve or worsen test performance; inspect both versions.


In [ ]:
forecast_summary = run_research(
    FORECAST_ROOT / "ACLED_Kenya_1997-2025.csv",
    FORECAST_ROOT / "artifacts/forecasting",
    coverage_end=FORECAST_COVERAGE_END,
    reporting_delay_days=REPORTING_DELAY_DAYS,
)
display(pd.Series(forecast_summary))
display(pd.read_csv(FORECAST_ROOT / "artifacts/forecasting/rolling_metrics.csv"))
display(pd.read_csv(FORECAST_ROOT / "artifacts/forecasting/annual_test_metrics.csv"))
display(pd.read_csv(FORECAST_ROOT / "artifacts/forecasting/county_test_metrics.csv"))
display(pd.read_csv(FORECAST_ROOT / "artifacts/forecasting/test_reliability.csv"))


## 16.2 Next deployment gate

Keep this research bundle separate from the event severity registry. Before live forecasting,
collect authenticated ACLED snapshots with first_seen_at and version history, verify complete
release coverage, evaluate reporting-delay sensitivity, define an alert review policy, and
validate performance prospectively. Never use the 2025 data to display a current 2026 risk map.
The first version intentionally retains the existing event-assessment dashboard.

References: [ACLED API](https://acleddata.com/api-documentation/getting-started),
[scikit-learn probability calibration](https://scikit-learn.org/stable/modules/calibration.html).
